# Lab - Multi-Turn Conversations

Three tasks that follow one customer conversation. You will start it, watch Claude
lose the thread, then give the thread back.

| Task | What you learn |
| --- | --- |
| 1 | Start the conversation and read the reply |
| 2 | Every API call starts from nothing |
| 3 | Your application owns the conversation |

**You do not need an API key.** This lab ships with a Claude simulator, so every cell
runs offline. The code you write is exactly the code you would write against the real
API - set `ANTHROPIC_API_KEY` at home and the same notebook calls Claude for real.

**You do not write code from scratch.** Each cell already holds the code, with blanks
marked `...` and a comment telling you what goes in each one.

One thing worth knowing before you start: every message in a conversation carries a
**role**, and there are only two. `"user"` is the customer, `"assistant"` is ShopAssist.
That is how Claude tells who said what.

## Setup

Run this cell first, before anything else. Click it, then press **Shift+Enter**.

In [ ]:
# --- Lab setup (provided - just run it) ---
import shopassist_lab
from shopassist_lab import check

# Everything below this line is ordinary Claude API code.
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()

model = "claude-sonnet-4-6"

# The two things the customer says, in order.
FIRST_MESSAGE = "I want to return my order."
ORDER_NUMBER_REPLY = "My order number is ORD-12345678."

---

## Task 1 - Start the conversation

In [ ]:
# ============================================================
# TASK 1 - Start the conversation
# ============================================================
#
# WHAT TO DO
#   Send the customer's opening message and read what
#   ShopAssist says back.
#
# WHY IT MATTERS
#   This is the first half of a two-turn conversation. Notice
#   that ShopAssist cannot help yet - it has to ask which order
#   the customer means. That question is what the next task
#   answers.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's send a simple support message" - about 3 minutes
#   3 seconds in.
#
# HOW TO DO IT
#   One blank. Send FIRST_MESSAGE as a single user message.
# ============================================================

# BEGIN SOLUTION
opening = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {"role": "user", "content": FIRST_MESSAGE},
    ],
)
# SCAFFOLD: opening = client.messages.create(
# SCAFFOLD:     model=model,
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     messages=[
# SCAFFOLD:         {"role": "user", "content": ...},   # the FIRST_MESSAGE variable
# SCAFFOLD:     ],
# SCAFFOLD: )
# END SOLUTION: replace the ... below with the value named beside it

print("Customer:   ", FIRST_MESSAGE)
print()
print("ShopAssist: ", opening.content[0].text)

check("opening_message", opening=opening)

---

## Task 2 - Answer the question, and lose the thread

In [ ]:
# ============================================================
# TASK 2 - Answer the question, and lose the thread
# ============================================================
#
# WHAT TO DO
#   The customer answers ShopAssist's question. Send only that
#   answer, exactly as a naive application would - one new
#   message, nothing in front of it.
#
# WHY IT MATTERS
#   The Claude API is stateless. It remembers nothing between
#   calls, so this request knows nothing about Task 1. You are
#   about to send a sentence that only makes sense in context,
#   to a model that has no context.
#
#   This is the mistake on purpose. Watch what comes back.
#
# WHERE TO SEE IT IN THE LECTURE
#   "But here is the mistake. We send only the new message" -
#   about 3 minutes 47 seconds in.
#
# HOW TO DO IT
#   One blank. Send ORDER_NUMBER_REPLY on its own.
# ============================================================

# BEGIN SOLUTION
without_history = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {"role": "user", "content": ORDER_NUMBER_REPLY},
    ],
)
# SCAFFOLD: without_history = client.messages.create(
# SCAFFOLD:     model=model,
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     messages=[
# SCAFFOLD:         {"role": "user", "content": ...},   # the ORDER_NUMBER_REPLY variable
# SCAFFOLD:     ],
# SCAFFOLD: )
# END SOLUTION: replace the ... below with the value named beside it

print("Customer:   ", ORDER_NUMBER_REPLY)
print()
print("ShopAssist: ", without_history.content[0].text)

check("no_history", without_history=without_history)

---

## Task 3 - Send it again, with the history

In [ ]:
# ============================================================
# TASK 3 - Send it again, with the history
# ============================================================
#
# WHAT TO DO
#   Send the exact same sentence as Task 2, but put the two
#   earlier turns in front of it: what the customer said, and
#   what ShopAssist replied.
#
# WHY IT MATTERS
#   Nothing about Claude changed between Task 2 and Task 3. The
#   only difference is what your application chose to send. In
#   production that history lives in your database or session
#   store, and your backend replays the relevant part on every
#   single request. Conversation memory is your job, not the
#   model's.
#
# WHERE TO SEE IT IN THE LECTURE
#   "Now let's send the same follow-up correctly" - about
#   4 minutes 36 seconds in.
#
# HOW TO DO IT
#   Three blanks. Two are the customer's words, which you
#   already have as variables.
#
#   The third is a role, and there are only two to choose from:
#
#       "user"        the customer speaking
#       "assistant"   ShopAssist speaking
#
#   The middle line is ShopAssist's reply from Task 1, so pick
#   the role that matches.
# ============================================================

# BEGIN SOLUTION
with_history = client.messages.create(
    model=model,
    max_tokens=300,
    messages=[
        {"role": "user", "content": FIRST_MESSAGE},
        {"role": "assistant", "content": "Sure, I can help with that. Could you share your order number?"},
        {"role": "user", "content": ORDER_NUMBER_REPLY},
    ],
)
# SCAFFOLD: with_history = client.messages.create(
# SCAFFOLD:     model=model,
# SCAFFOLD:     max_tokens=300,
# SCAFFOLD:     messages=[
# SCAFFOLD:         {"role": "user", "content": ...},   # the FIRST_MESSAGE variable
# SCAFFOLD:         {"role": ..., "content": "Sure, I can help with that. Could you share your order number?"},   # "user" or "assistant"? ShopAssist said this
# SCAFFOLD:         {"role": "user", "content": ...},   # the ORDER_NUMBER_REPLY variable
# SCAFFOLD:     ],
# SCAFFOLD: )
# END SOLUTION: replace each ... below with the value named beside it

print("WITHOUT history:")
print(without_history.content[0].text)
print()
print("WITH history:")
print(with_history.content[0].text)

check("with_history", without_history=without_history, with_history=with_history)

---

## Done

Same final sentence in Task 2 and Task 3, two completely different answers. The
conversation is not stored inside Claude - it is stored by you, and sent again on
every request.

In the next lab you stop writing these message dictionaries by hand.